# DSFB-GPU — Public S-REAL Replay Notebook (COLAB.S-REAL.1)

This notebook **rebuilds the CUDA path from source** and **re-runs the
20-dataset S-REAL audit gauntlet** on the panel-locked vendored
fixtures. It is a public replay artifact, not a synthetic-fixture
demo. The notebook produces a downloadable ZIP bundle containing all
180 audit artifacts, the sealed bundle receipts, and the operator
navigation INDEX.

**Expected runtime on a free Colab GPU runtime**: ~3–5 minutes
end-to-end (CUDA build ~90–120 s; audit run ~30–90 s; bundle pack ~1 s).

**Hardware requirement**: any Colab GPU runtime (T4 / A100 / L4).
CPU-only Colab is **not supported** because the `s-real-audit` driver
requires the CUDA path.

## Panel-locked honest framing

> The notebook attempts cross-hardware replay on Colab T4 / A100 / L4.
> If the emitted artifacts match the committed S-REAL bundle, this is
> evidence of cross-hardware determinism for that environment. If they
> diverge, the notebook reports the exact differing artifact and
> preserves the result as an honest portability finding.

## Panel-locked non-claims

- This notebook does NOT add new datasets, kernels, or hashes.
- This notebook does NOT change S-REAL.3.1.1's claims; it only makes
  them reproducible on third-party hardware.
- This notebook does NOT claim Colab-runtime saturation or any new
  performance result.
- This notebook does NOT claim cross-hardware byte-identity until the
  F-gate has actually been run on this run.
- The notebook is offered as a reproducibility tool, NOT a benchmarking
  platform — Colab thermal variance makes any per-run GB/s number a
  courtesy snapshot, not a measurement claim.


## How to use this notebook

1. Open **this notebook** in Colab.
2. Set **Runtime → Change runtime type → GPU** (T4 is sufficient;
   A100 / L4 also work).
3. **Run all cells** top-to-bottom (Runtime → Run all). The first
   setup cell downloads the checked-in `dsfb-gpu.tar.gz` source
   bundle from the GitHub `main` branch and extracts it automatically.
4. The final cell triggers a browser download of the audit bundle ZIP.

If any cell halts, scroll up to the printed error message — every
failure path is paneled to be self-describing.


## §1. Environment report + source bundle

This section downloads the checked-in source tarball, extracts it, and prints the
hardware + toolchain identifiers so the bundle's receipt records
exactly which environment produced the artifacts.


In [ ]:
import urllib.request
from pathlib import Path

TARBALL = Path("dsfb-gpu.tar.gz")
RAW_TARBALL_URL = "https://raw.githubusercontent.com/infinityabundance/dsfb/main/crates/dsfb-gpu/notebooks/dsfb-gpu.tar.gz"

if not TARBALL.exists():
    print("downloading built-in source bundle...")
    urllib.request.urlretrieve(RAW_TARBALL_URL, TARBALL)
else:
    print("using existing source bundle:", TARBALL)

print("tarball:", TARBALL, TARBALL.stat().st_size, "bytes")


In [ ]:
%%bash
set -euo pipefail
rm -rf dsfb-gpu
tar -xzf dsfb-gpu.tar.gz
cd dsfb-gpu
pwd
python3 - <<'PY'
import os
for name in sorted(os.listdir('.'))[:20]:
    print(name)
PY


In [ ]:
%%bash
set -euo pipefail
cd dsfb-gpu
echo "=== nvidia-smi ==="
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader || echo "(nvidia-smi not available)"
echo ""
echo "=== nvcc ==="
nvcc --version | tail -4 || echo "(nvcc not on PATH)"
echo ""
echo "=== rustc ==="
if command -v rustc >/dev/null 2>&1; then
    rustc --version
else
    echo "(installing stable Rust toolchain — first-time only)"
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
    source "$HOME/.cargo/env"
    rustc --version
fi
echo ""
echo "=== uname ==="
uname -srm


## §2. Build the CUDA binary

`cargo build --release --features cuda -p dsfb-gpu-debug-demo` builds
the `dsfb-gpu-debug` binary that the audit driver lives in. The build
script (`crates/dsfb-gpu-debug-cuda/build.rs`) locates `nvcc`,
compiles `cuda/kernels.cu` with `--fmad=false --use_fast_math=false`
(the panel-locked determinism flags), and links the static archive.

**Expected build time**: ~90–120 s clean. The build prints the last
20 lines to keep the output panel readable; the full log is in the
cell's output history if you need to scroll.

**Stop condition**: if nvcc is not on PATH, the build will fail
with a clear error. There is no CPU fallback for the audit — it
requires the CUDA path by construction.


In [ ]:
%%bash
set -euo pipefail
# Pick up Rust if the toolchain was just installed in §1.
source "$HOME/.cargo/env" 2>/dev/null || true
cd dsfb-gpu
echo "=== building dsfb-gpu-debug with --features cuda (release) ==="
time cargo build --release --features cuda -p dsfb-gpu-debug-demo 2>&1 | tail -20
echo ""
echo "=== binary ==="
ls -la target/release/dsfb-gpu-debug


## §3. Dataset SHA-256 verification (B-gate)

Walk every entry in `data/fixtures/MANIFEST.toml`; recompute SHA-256
on the live TSV file; compare against the manifest's pinned hash.

The audit driver enforces this same check at dispatch time (refuses
to admit a TSV whose SHA mismatches the pin), so a failure here means
the upload or extraction corrupted bytes. PASS evidence here is what
the bundle's B-gate verdict claims.


In [ ]:
import hashlib
import os
import re

repo_root = "dsfb-gpu"
manifest_path = os.path.join(repo_root, "data", "fixtures", "MANIFEST.toml")

# Lightweight TOML reader: we only need (id, path, sha256, license)
# tuples; full TOML parsing isn't worth a dependency in a notebook.
with open(manifest_path, "r", encoding="utf-8") as f:
    text = f.read()

raw_fixtures = []
for block in re.finditer(
    r"\[fixtures\.([a-zA-Z0-9_]+)\][^\[]*?path\s*=\s*\"([^\"]+)\"[^\[]*?sha256\s*=\s*\"([^\"]+)\"[^\[]*?(?:license\s*=\s*\"([^\"]+)\")?",
    text, re.DOTALL,
):
    raw_fixtures.append({
        "id": block.group(1),
        "path": block.group(2),
        "pinned_sha": block.group(3),
        "license": block.group(4) or "(unspecified)",
    })

# Partition fixtures into:
#   - audit_required: the 20 small TSVs the audit consumes (path lives
#     directly under data/fixtures/, NOT a *_1024x1024.tsv or
#     *_512x1024.tsv saturation variant)
#   - excluded_on_colab: upstream archives (path starts with `../`)
#     and saturation-class TSVs (the slim Colab tarball drops them
#     because the audit doesn't need them and they add ~100 MB)
#
# B-gate evaluates the audit_required set; the excluded set is
# reported transparently so the operator knows what is intentionally
# absent on Colab.
def is_excluded_on_colab(path):
    # Excluded on Colab tarball (per pack_for_colab.sh):
    #   - upstream archives (paths starting with ../)
    #   - saturation-class TSVs (any *x1024.tsv stem — covers
    #     1024x1024 standard, imdb_tgz's 1020x1024, deepsense6g's
    #     512x1024)
    if path.startswith("../"):
        return True
    if path.endswith("x1024.tsv"):
        return True
    return False

audit_required = [fx for fx in raw_fixtures if not is_excluded_on_colab(fx["path"])]
excluded = [fx for fx in raw_fixtures if is_excluded_on_colab(fx["path"])]

print(f"manifest fixtures parsed: {len(raw_fixtures)}")
print(f"  audit-required (Colab):     {len(audit_required)}")
print(f"  excluded on Colab tarball:  {len(excluded)}  (upstream archives + saturation TSVs)")
print()

passed = 0
failed = []
missing = []
for fx in audit_required:
    full = os.path.join(repo_root, "data", "fixtures", fx["path"])
    if not os.path.isfile(full):
        missing.append(fx["id"])
        continue
    h = hashlib.sha256()
    with open(full, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    live_sha = h.hexdigest()
    if live_sha == fx["pinned_sha"]:
        passed += 1
    else:
        failed.append((fx["id"], fx["pinned_sha"][:16], live_sha[:16]))

print(f"PASS: {passed} / {len(audit_required)} audit-required fixtures")
if missing:
    print(f"MISSING ({len(missing)}): {missing}")
if failed:
    print(f"FAILED ({len(failed)}):")
    for fid, pinned_prefix, live_prefix in failed:
        print(f"  {fid:30s}  pinned={pinned_prefix}...  live={live_prefix}...")

B_PASS = (passed == len(audit_required) and not failed and not missing and len(audit_required) >= 20)
print()
print(f"B-gate (dataset SHA verification): {'PASS' if B_PASS else 'FAIL'}")


## §4. Run the 20-dataset S-REAL audit (C + D gates)

`dsfb-gpu-debug s-real-audit --dataset all --out-dir reports` walks
the **20 sealed audit datasets** (the `AUDIT_DATASETS` table in
`crates/dsfb-gpu-debug-demo/src/cli/s_real_audit.rs`) and emits 9
panel-locked artifacts per dataset to
`reports/<tier_dir>/<dataset_id>/` where `tier_dir` is `s_real_1` /
`s_real_2` / `s_real_3` per the sealed bundle layout pinned in
`reports/s_real_3/bundle_manifest.toml`.

**Scope-boundary discipline (panel-locked S-REAL.3.1.2)**: `--dataset
all` enumerates ONLY the 20 audit datasets. The 10 saturation-class
fixtures (RadioML / DeepBeam / RadioML-Gold / POWDER / ORACLE /
Deepsense6G / IMDb tarball / IMDb DuckDB / Snowset / SQLShare;
declared in the `SATURATION_FIXTURES` table) are NEVER selected by
`--dataset all`. They dispatch only via explicit single-id audit
calls (e.g. `s-real-audit --dataset radioml_2018_snr30_large`) and
via `scripts/s_real_saturation_sweep.sh`. Their TSVs (~90 MB total)
are deliberately excluded from the slim Colab tarball by
`scripts/pack_for_colab.sh` because the audit gauntlet's 20-dataset
replay does not need them.

The driver runs each dataset's dispatcher **twice** and asserts
byte-identical casefile / episodes output between the two runs (the
D-gate / per-run replay-verification gate). If any dataset's two
dispatches diverge, the driver exits non-zero.

The audit also enforces the SHA-pin at dispatch time (refuses to
admit a TSV whose SHA mismatches `MANIFEST.toml`). Cell §3 already
verified this; the audit re-verifies as belt-and-braces.

**Expected runtime**: ~30–90 s on T4.

In [ ]:
%%bash
set -euo pipefail
source "$HOME/.cargo/env" 2>/dev/null || true
cd dsfb-gpu
echo "=== running s-real-audit on all 20 datasets ==="
time ./target/release/dsfb-gpu-debug s-real-audit --dataset all --out-dir reports 2>&1 | tail -50
echo ""
echo "=== per-tier dataset counts ==="
ls -la reports/s_real_1/ reports/s_real_2/ reports/s_real_3/ 2>/dev/null | grep '^d' | grep -v '\.$' | wc -l
echo "(20 dataset directories expected)"


## §5. Pack audit artifacts into a downloadable ZIP

`scripts/package_s_real_colab_outputs.sh` collects every operator-
facing piece into a single ZIP under `/content/dsfb_gpu_audit_bundle.zip`:
all three tier directories (180 audit artifacts + sealed bundle
receipts), `reports/INDEX.md` (operator navigation root), the
saturation-sweep classification (if present), and
`colab_run_receipt.md` (this run's provenance).

The script refuses to build the ZIP if any of the three tier
directories is missing — partial bundles are misleading.


In [ ]:
%%bash
set -euo pipefail
cd dsfb-gpu
bash scripts/package_s_real_colab_outputs.sh --out /content/dsfb_gpu_audit_bundle.zip
echo ""
echo "=== ZIP contents (first 20 entries) ==="
python3 - <<'PY'
import zipfile
path = "/content/dsfb_gpu_audit_bundle.zip"
print("Archive:", path)
with zipfile.ZipFile(path) as zf:
    for info in zf.infolist()[:20]:
        print(f"{info.file_size:10d} {info.filename}")
PY


## §6. Bundle integrity + per-run replay verification (E + D surfaces)

The bundle-integrity test recomputes SHA-256 over every file in the
60-row hash chain (`reports/s_real_3/bundle_hash_chain.txt`) and
asserts each matches the committed bundle's pinned hash. If the
Colab-generated bytes are byte-identical to the dev-machine seal,
**E PASSES** (E-gate). Divergence at any artifact is the **F-gate**'s
evidence and is surfaced explicitly — the notebook treats this as a
FINDING, not a crash, per the panel-locked honest-framing rule.

The cell after this one walks each dataset's `replay_verification.txt`
and prints the per-dataset within-run replay verdict (D-gate
evidence).


In [ ]:
%%bash
set +e   # we want to read the test output even on FAIL — divergence is a finding
source "$HOME/.cargo/env" 2>/dev/null || true
cd dsfb-gpu
echo "=== running s_real_3_bundle_integrity test (5 tests) ==="
cargo test --release --features cuda --test s_real_3_bundle_integrity -- --nocapture 2>&1 | tail -30
echo ""
echo "(exit 0: E-gate PASS + F-gate PASS for the 60-row chain;"
echo " exit non-0: at least one chain row diverged — see assertion output above"
echo " for the exact artifact + dataset)"


In [ ]:
import os

repo_root = "dsfb-gpu"
tiers = ["s_real_1", "s_real_2", "s_real_3"]

per_dataset = []
for tier in tiers:
    tier_dir = os.path.join(repo_root, "reports", tier)
    if not os.path.isdir(tier_dir):
        continue
    for dataset_id in sorted(os.listdir(tier_dir)):
        ds_dir = os.path.join(tier_dir, dataset_id)
        if not os.path.isdir(ds_dir):
            continue
        rv = os.path.join(ds_dir, "replay_verification.txt")
        if os.path.isfile(rv):
            body = open(rv, "r", encoding="utf-8").read()
            replay_yes = "byte-identical replay: YES" in body
        else:
            replay_yes = False
        per_dataset.append((tier, dataset_id, replay_yes))

print(f"per-dataset replay verification ({len(per_dataset)} datasets):")
print()
all_yes = True
for tier, dataset_id, yes in per_dataset:
    flag = "YES" if yes else "NO "
    if not yes:
        all_yes = False
    print(f"  {tier:10s} | {dataset_id:30s} | replay: {flag}")
print()
D_PASS = (all_yes and len(per_dataset) == 20)
print(f"D-gate (per-run replay): {'PASS' if D_PASS else 'FAIL (or partial)'}")


## §7. Result summary — A–F classification

Per the panel-locked classification:

- **A — Build success**: §2 cargo build completed.
- **B — Dataset SHA verification**: §3 (20 / 20 fixtures match
  pinned SHA).
- **C — Audit run success**: §4 (180 artifacts emitted, exit 0).
- **D — Per-run replay**: every dataset's two dispatches produced
  byte-identical casefile + episodes (within-run determinism).
- **E — Bundle integrity**: §6 cargo test passed; the 60-row hash
  chain matches Colab-generated bytes.
- **F — Cross-hardware byte-identity**: this run's bytes are
  byte-identical to the dev-machine seal recorded in the committed
  bundle. F is independent of E; E asserts the chain is internally
  consistent, F asserts the cross-hardware bytes match. In practice
  the bundle-integrity test fires the same gate, so E PASS implies
  F PASS for the 60 chain-pinned artifacts.

A–F can pass independently. A successful audit (A–E PASS) with F
DIVERGENT is still a valuable finding — it documents a portability
boundary at the exact artifact path.


In [ ]:
import os
import json
import hashlib
import re

repo_root = "dsfb-gpu"

# A — Build success (binary exists)
binary = os.path.join(repo_root, "target", "release", "dsfb-gpu-debug")
A_PASS = os.path.isfile(binary)

# B — Dataset SHA verification (audit-required subset only; see §3 for
# the exclusion rule that drops upstream archives + saturation TSVs)
def is_excluded_on_colab(path):
    # see §3 for the per-Colab exclusion rule (drop upstream
    # archives and saturation-class *x1024.tsv variants)
    return path.startswith("../") or path.endswith("x1024.tsv")
manifest = open(os.path.join(repo_root, "data", "fixtures", "MANIFEST.toml")).read()
B_required = 0
B_pass = 0
for block in re.finditer(
    r"\[fixtures\.([a-zA-Z0-9_]+)\][^\[]*?path\s*=\s*\"([^\"]+)\"[^\[]*?sha256\s*=\s*\"([^\"]+)\"",
    manifest, re.DOTALL,
):
    path, pinned = block.group(2), block.group(3)
    if is_excluded_on_colab(path):
        continue
    B_required += 1
    full = os.path.join(repo_root, "data", "fixtures", path)
    if not os.path.isfile(full):
        continue
    h = hashlib.sha256()
    with open(full, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    if h.hexdigest() == pinned:
        B_pass += 1
B_PASS = (B_pass == B_required and B_required >= 20)

# C — Audit run success (180 artifacts across the 3 tiers)
def count_artifacts(tier):
    tier_dir = os.path.join(repo_root, "reports", tier)
    if not os.path.isdir(tier_dir):
        return 0
    count = 0
    for ds in os.listdir(tier_dir):
        ds_dir = os.path.join(tier_dir, ds)
        if os.path.isdir(ds_dir):
            count += sum(1 for f in os.listdir(ds_dir) if os.path.isfile(os.path.join(ds_dir, f)))
    return count
C_artifacts = (
    count_artifacts("s_real_1")
    + count_artifacts("s_real_2")
    + count_artifacts("s_real_3")
)
C_PASS = (C_artifacts >= 180)  # 9 × 20 = 180

# D — Per-run replay (every dataset's replay_verification.txt says YES)
per_dataset = []
for tier in ["s_real_1", "s_real_2", "s_real_3"]:
    tier_dir = os.path.join(repo_root, "reports", tier)
    if not os.path.isdir(tier_dir):
        continue
    for dataset_id in sorted(os.listdir(tier_dir)):
        ds_dir = os.path.join(tier_dir, dataset_id)
        if not os.path.isdir(ds_dir):
            continue
        rv = os.path.join(ds_dir, "replay_verification.txt")
        replay_yes = (
            os.path.isfile(rv)
            and "byte-identical replay: YES" in open(rv, "r", encoding="utf-8").read()
        )
        per_dataset.append((tier, dataset_id, ds_dir, replay_yes))
D_PASS = (len(per_dataset) == 20 and all(y for _, _, _, y in per_dataset))

# E + F — bundle hash chain comparison (Colab bytes vs committed seal)
chain_path = os.path.join(repo_root, "reports", "s_real_3", "bundle_hash_chain.txt")
EF_failures = []
EF_pass_count = 0
EF_total = 0
if os.path.isfile(chain_path):
    chain = open(chain_path).read()
    for line in chain.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split(None, 1)
        if len(parts) != 2 or len(parts[0]) != 64:
            continue
        pinned_sha, rel = parts[0], parts[1]
        full = os.path.join(repo_root, rel)
        EF_total += 1
        if not os.path.isfile(full):
            EF_failures.append((rel, "MISSING"))
            continue
        h = hashlib.sha256()
        with open(full, "rb") as f:
            for chunk in iter(lambda: f.read(65536), b""):
                h.update(chunk)
        live = h.hexdigest()
        if live == pinned_sha:
            EF_pass_count += 1
        else:
            EF_failures.append((rel, f"pinned={pinned_sha[:16]} live={live[:16]}"))
E_PASS = (EF_failures == [] and EF_total >= 60)
F_PASS = E_PASS  # for the 60 chain-pinned artifacts E and F coincide

print("COLAB.S-REAL.1 verdict:")
print(f"  A  Build success                              : {'PASS' if A_PASS else 'FAIL'}")
print(f"  B  Dataset SHA verification success           : {'PASS' if B_PASS else 'FAIL'}  ({B_pass}/{B_required} audit-required)")
print(f"  C  Audit run success                          : {'PASS' if C_PASS else 'FAIL'}  ({C_artifacts} artifacts emitted)")
print(f"  D  Per-run replay success                     : {'PASS' if D_PASS else 'FAIL'}  ({sum(1 for _,_,_,y in per_dataset if y)}/{len(per_dataset)})")
print(f"  E  Bundle integrity success                   : {'PASS' if E_PASS else 'FAIL'}  ({EF_pass_count}/{EF_total} chain rows)")
print(f"  F  Cross-hardware byte-identity success       : {'PASS' if F_PASS else 'DIVERGENT'}")
if EF_failures:
    print()
    print("F-gate divergence detail (first 5):")
    for rel, msg in EF_failures[:5]:
        print(f"  {rel}: {msg}")
print()

# Per-dataset summary table
print("Per-dataset summary:")
print(f"  {'dataset_id':30s} | {'events':>8s} | {'episodes':>9s} | {'replay':>6s} | casefile_final_hash")
print(f"  {'-'*30}-+-{'-'*8}-+-{'-'*9}-+-{'-'*6}-+-{'-'*16}")
total_events = 0
total_episodes = 0
for tier, dataset_id, ds_dir, replay_yes in per_dataset:
    cf_path = os.path.join(ds_dir, "casefile.json")
    events = 0
    episodes = 0
    final_hash = ""
    if os.path.isfile(cf_path):
        try:
            cf = json.load(open(cf_path))
            episodes = len(cf.get("episodes", []))
            events = cf.get("n_events", 0) or cf.get("event_count", 0)
            final_hash = (cf.get("final_case_file_hash", "") or "")[:16]
        except Exception:
            pass
    rrp = os.path.join(ds_dir, "run_receipt.txt")
    if os.path.isfile(rrp) and events == 0:
        for line in open(rrp).read().splitlines():
            if "n_events" in line or "events_lowered" in line:
                m = re.search(r"(\d+)", line)
                if m:
                    events = int(m.group(1)); break
    total_events += events
    total_episodes += episodes
    print(f"  {dataset_id:30s} | {events:>8d} | {episodes:>9d} | {'YES' if replay_yes else 'NO ':>6s} | {final_hash}")
print()
print(f"Aggregate: 20 datasets / {total_events} events / {total_episodes} episodes / "
      f"{sum(1 for _,_,_,y in per_dataset if y)} replay-YES")
print()
print(f"Panel-locked bundle headline: 20 datasets / 316 episodes / 5 source-class families")
print(f"  this run admitted {total_episodes} episodes (matches headline: {total_episodes == 316})")


In [ ]:
# S-REAL.3.1.2 PENDING-guard: populate colab_run_receipt.md inside the
# bundle ZIP with the verdict computed in §7, then assert no
# <PENDING> markers remain anywhere in the bundle's text artifacts.
#
# WHY: the package script (scripts/package_s_real_colab_outputs.sh)
# emits the receipt with <PENDING> placeholders for the A-F gates
# and the per-dataset table. The notebook is responsible for
# substituting the populated body before files.download(). Before
# S-REAL.3.1.2 this overwrite was missing — operators would receive
# a ZIP whose colab_run_receipt.md still said <PENDING> on every
# gate, undermining the artifact's credibility. This cell closes
# the gap and the assertion at the bottom prevents the regression.
import zipfile
import shutil
import os
import json

ZIP_PATH = "/content/dsfb_gpu_audit_bundle.zip"
RECEIPT_BASENAME = "colab_run_receipt.md"

def fmt_pass(p):
    return "PASS" if p else "FAIL"

def fmt_replay(yes):
    return "YES" if yes else "NO"

# Build per-dataset table rows from the §7 variables.
rows = []
for tier, dataset_id, ds_dir, replay_yes in per_dataset:
    n_events = 0
    n_episodes = 0
    final_hash16 = ""
    cf_path = os.path.join(ds_dir, "casefile.json")
    if os.path.isfile(cf_path):
        try:
            cf = json.load(open(cf_path))
            n_episodes = len(cf.get("episodes", []))
            n_events = cf.get("n_events", 0) or cf.get("event_count", 0)
            final_hash16 = (cf.get("final_case_file_hash", "") or "")[:16]
        except Exception:
            pass
    rows.append(
        f"| {dataset_id} | {n_events} | {n_episodes} | "
        f"{fmt_replay(replay_yes)} | {final_hash16} |"
    )
per_dataset_table = "\n".join(rows) if rows else "| (no datasets) | | | | |"

populated_receipt = f"""# COLAB.S-REAL.1 — Run Receipt (populated by notebook §7)

This receipt was populated by the notebook's PENDING-guard cell after
the §7 A-F classification verdict was computed. The packer
(`scripts/package_s_real_colab_outputs.sh`) emitted the skeleton with
`<PENDING>` markers; the notebook substituted the populated body
before inviting the operator to download.

## Honest framing (panel-locked verbatim)

The notebook attempts cross-hardware replay on Colab T4 / A100 / L4.
If the emitted artifacts match the committed S-REAL bundle, this is
evidence of cross-hardware determinism for that environment. If they
diverge, the notebook reports the exact differing artifact and
preserves the result as an honest portability finding.

## A-F classification

```
A  Build success                              : {fmt_pass(A_PASS)}
B  Dataset SHA verification success           : {fmt_pass(B_PASS)}  ({B_pass}/{B_required} audit-required)
C  Audit run success                          : {fmt_pass(C_PASS)}  ({C_artifacts} artifacts emitted)
D  Per-run replay success                     : {fmt_pass(D_PASS)}  ({sum(1 for _,_,_,y in per_dataset if y)}/{len(per_dataset)})
E  Bundle integrity success                   : {fmt_pass(E_PASS)}  ({EF_pass_count}/{EF_total} chain rows)
F  Cross-hardware byte-identity success       : {'PASS' if F_PASS else 'DIVERGENT'}
```

## Per-dataset summary

| dataset_id | events | admitted_episodes | replay_verified | casefile_final_hash[:16] |
|------------|-------:|------------------:|:---------------:|:--------------------|
{per_dataset_table}

## Aggregate

- total_datasets      : 20
- total_events        : {total_events}
- total_episodes      : {total_episodes}  (panel-locked headline: 316)
- replay_verified_yes : {sum(1 for _,_,_,y in per_dataset if y)} / 20

## Panel-locked non-claims (verbatim, MUST appear)

- COLAB.S-REAL.1 does NOT add new datasets, kernels, or hashes.
- COLAB.S-REAL.1 does NOT change S-REAL.3.1.1's claims; it only
  makes them reproducible on third-party hardware.
- COLAB.S-REAL.1 does NOT claim Colab-runtime saturation or any new
  performance result.
- COLAB.S-REAL.1 does NOT claim cross-hardware byte-identity until
  the F-gate has actually been run; the notebook reports what it
  observed honestly.
- COLAB.S-REAL.1 does NOT modify the audit's algorithm, fixtures,
  contracts, or hash chain.
- Colab T4 / A100 / L4 is NOT RTX 4080 SUPER + CUDA 13.2 (the
  dev-machine hardware anchor); timing values in `perf_profile.txt`
  are RUNTIME-DEPENDENT and NOT comparable to the anchor.
"""

# Substitute the receipt inside the ZIP via tmp + rename so the swap
# is atomic from the operator's point of view.
tmp_path = ZIP_PATH + ".tmp"
substituted = False
with zipfile.ZipFile(ZIP_PATH, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        body = zin.read(item.filename)
        if item.filename.endswith("/" + RECEIPT_BASENAME) or item.filename == RECEIPT_BASENAME:
            body = populated_receipt.encode("utf-8")
            substituted = True
        zout.writestr(item, body)
shutil.move(tmp_path, ZIP_PATH)

if not substituted:
    print(f"WARNING: no {RECEIPT_BASENAME} entry found inside ZIP {ZIP_PATH}; "
          "skipping substitution (the packer may have changed paths).")

# PENDING-guard: re-open the ZIP and assert no <PENDING> markers
# remain in any text artifact. This is the load-bearing assertion
# the S-REAL.3.1.2 hygiene seal adds.
pending_hits = []
with zipfile.ZipFile(ZIP_PATH, "r") as zin:
    for item in zin.namelist():
        if not item.endswith((".md", ".txt", ".html", ".jsonl", ".toml")):
            continue
        try:
            body = zin.read(item).decode("utf-8", errors="ignore")
        except Exception:
            continue
        if "<PENDING>" in body:
            pending_hits.append(item)

if pending_hits:
    raise AssertionError(
        f"PENDING-guard FAILED: {len(pending_hits)} text artifact(s) inside "
        f"{ZIP_PATH} still contain <PENDING> markers; the §7 verdict cell "
        f"must populate every gate before files.download(). First hits: "
        f"{pending_hits[:5]}"
    )

print(f"PENDING-guard PASS: no <PENDING> markers in any text artifact "
      f"of {ZIP_PATH}; safe to download.")


## §8. Non-claims + citation pointers

This notebook is a **reproducibility tool**, NOT a benchmarking
platform. Repeating verbatim (because every public artifact MUST
carry these):

- COLAB.S-REAL.1 does NOT add new datasets, kernels, or hashes.
- COLAB.S-REAL.1 does NOT change S-REAL.3.1.1's claims; it only
  makes them reproducible on third-party hardware.
- COLAB.S-REAL.1 does NOT claim Colab-runtime saturation or any
  new performance result — the saturation sweep is the dev-machine
  hardware-anchored measurement.
- COLAB.S-REAL.1 does NOT claim cross-hardware byte-identity until
  the F-gate has actually been run; this run's A–F verdict is the
  honest report.
- COLAB.S-REAL.1 does NOT modify the audit's algorithm, fixtures,
  contracts, or hash chain.
- The downloaded bundle does NOT claim Zenodo deposit or DOI
  assignment on its own; those are separate publication acts.
- Colab T4 / A100 / L4 is NOT RTX 4080 SUPER + CUDA 13.2 (the
  dev-machine hardware anchor for S-PERF.16.a and the saturation-
  sweep numbers); timing values in `perf_profile.txt` are
  RUNTIME-DEPENDENT and NOT comparable to the anchor.

### Citation pointers

- Sealed commit chain (most recent first):
  - `3fdf42f`  S-REAL.3.1.1 hygiene close-out
  - `fde8a99`  S-REAL.3.1 bundle integrity gate + saturation sweep
  - `a8aaa04`  S-REAL.3 20-dataset sealed; Zenodo-publishable bundle
- The bundle's `INDEX.md` is the operator navigation root.


In [ ]:
from google.colab import files
files.download('/content/dsfb_gpu_audit_bundle.zip')
